In [ ]:
from google.colab import files

files.upload()

Saving data.zip to data.zip
Buffered data was truncated after reaching the output size limit.

In [ ]:
import zipfile

zip_path = "/content/data.zip"
extract_path = "/content/cifake"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")




Dataset extracted successfully!


In [ ]:
import os

os.listdir("/content/cifake")

['train', 'test']

In [3]:
from torchvision import transforms

train_tf = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print("transforms successfull")

transforms successfull


In [4]:
from torchvision import datasets
from torch.utils.data import DataLoader

train_dir = "/content/cifake/train"
test_dir = "/content/cifake/test"

train_dataset = datasets.ImageFolder(train_dir, transform=train_tf)
test_dataset = datasets.ImageFolder(test_dir, transform=train_tf)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("datasets successfull")
print("Classes:", train_dataset.classes)
print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

datasets successfull
Classes: ['FAKE', 'REAL']
Training images: 100000
Testing images: 20000


In [5]:
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 3, 300, 300])
Label batch shape: torch.Size([32])


In [6]:
!pip install transformers timm -q

In [7]:
import timm

model = timm.create_model("efficientnet_b3", pretrained=False, num_classes=2)

print("model loaded successfully")

model loaded successfully


In [8]:
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

print("loss function:", loss_function)
print("device:", device)

loss function: CrossEntropyLoss()
device: cpu


In [9]:
from tqdm import tqdm
from torch.optim.lr_scheduler import OneCycleLR

num_epochs = 5

scheduler = OneCycleLR(
    optimizer,
    max_lr=1,
    steps_per_epoch=len(train_loader),
    epochs=5
)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Fix: Directly assign the model's output to logits
        logits = model(images)

        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(logits, dim=1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

        loop.set_postfix(loss=loss.item(), accuracy=100*correct/total)

        scheduler.step()

    print(f"Epoch {epoch+1} Training Loss: {total_loss/len(train_loader):.4f}")
    print(f"Epoch {epoch+1} Training Accuracy: {100*correct/total:.2f}%")

Epoch 1/5:   0%|          | 1/3125 [01:21<70:42:08, 81.48s/it, accuracy=50, loss=1.63]


KeyboardInterrupt: 